In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

해결 하고자 하는 문제 = 재방문율
방향성을 가지고 데이터 탐색


In [ ]:
#필요없는 컬럼
drop_cols_df2 = [
    'browser_version', 'email', 'geo_source', 'name',
    'initial_utm_campaign', 'initial_utm_content', 'initial_utm_id',
    'initial_utm_medium', 'initial_utm_source', 'initial_utm_term',
    'initial_utm_source_platform'
]

drop_cols_df3 = [
    'browser_is_whale', 'browser_is_chrome', 'browser_is_edge',
    'email', 'name', 'browser_version', 'insert_id',
    'screen_height', 'screen_width', 'search_engine', 'lib_version',
    'mp_api_endpoint', 'mp_lib', 'mp_loader', 'extension_version',
    'llm_provider', 'llm_version', 'position', 'platform', 'trigger'
]

1번 데이터셋 불러오기  

In [ ]:
df1 = pd.read_json('../data/raw/1. 사용자별 사용량 데이터.json')

df1 데이터프레임의 '_id' 컬럼이 MongoDB 스타일의 ObjectID 형식   
JSON 파일을 pandas로 읽어들일 때, 이런 중첩된 딕셔너리 구조가 그대로 로드 하는것을 방지   
df1의 '_id' 컬럼을 문자열로 변환 후, '_id_str' 컬럼에 저장  
'recent_execution_date'날짜 값을 추출한 후 datetime 형식으로 변환  

In [ ]:
df1['_id_str'] = df1['_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else None)
df1['recent_execution_date'] = df1['recent_execution_date'].apply(lambda x: x.get('$date') if isinstance(x, dict) else None)
df1['recent_execution_date'] = pd.to_datetime(df1['recent_execution_date'])

2번 데이터셋 불러오기  

In [ ]:
df2 = pd.read_json('data/2. 사용자 클라이언트 데이터.json', lines=True) 

In [ ]:
df2['user_id'].value_counts()

df2에 'properties' 컬럼을 중첩된 JSON으로 정규화  
'distinct_id'가 중복되면 삭제하고, 원본 df2와 병합해서 충돌 방지  
중첩된 속성을 평탄화  

In [ ]:
if 'properties' in df2.columns:
    properties_df = pd.json_normalize(df2['properties'])
    if 'distinct_id' in properties_df.columns:
        properties_df = properties_df.drop('distinct_id', axis=1)  
    df2 = pd.concat([df2.drop('properties', axis=1), properties_df], axis=1)

3번 데이터셋 불러오기

In [ ]:
df3 = pd.read_json('data/3. 이벤트 데이터.json', lines=True)

df3의 'user_id' 컬럼을 문자열로 변환 후 'user_id_str' 컬럼에 저장  
딕셔너리 형식에서 '$oid' 값을 추출

In [ ]:
df3['user_id_str'] = df3['user_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else x)

df3에 'properties' 컬럼이 있으면 이를 정규화  
'distinct_id'와 'user_id'가 중복되면 삭제, 원본 df3와 병합  
df2와 유사한 평탄화  

In [ ]:
if 'properties' in df3.columns:
    props_df3 = pd.json_normalize(df3['properties'])
    for col in ['distinct_id', 'user_id']:
        if col in props_df3.columns:
            props_df3 = props_df3.drop(col, axis=1)
    df3 = pd.concat([df3.drop('properties', axis=1), props_df3], axis=1)

In [ ]:
df3

df1과 df2를 병합하여 user_merged 데이터프레임을 만듬  
'_id_str'과 'distinct_id'를 키로 사용하며, outer join으로 모든 레코드를 포함  
사용자 사용량과 클라이언트 데이터를 결합  

In [ ]:
user_merged = pd.merge(df1, df2, left_on='_id_str', right_on='distinct_id', how='outer')

In [ ]:
user_merged

df3과 user_merged에 병합 키('merge_key')를 생성(병합할때 안정성 확보)  
'user_id_str'을 사용  
left join으로 병합 후 full_merged를 만들고 불필요한 키 컬럼을 삭제  
이벤트 데이터와 사용자 데이터를 통합  

In [ ]:
df3['merge_key'] = df3['user_id_str'].fillna(df3['distinct_id'])
user_merged['merge_key'] = user_merged['_id_str'].fillna(user_merged['distinct_id'])
full_merged = pd.merge(df3, user_merged, on='merge_key', how='left', suffixes=('_event', '_user'))
full_merged.drop(['merge_key_event', 'merge_key_user', 'merge_key'], axis=1, errors='ignore', inplace=True)

In [ ]:
full_merged

중복된 컬럼 이름을 검사, 중복 시 숫자 접미사를 추가해서 고유화  
해시 가능한 컬럼를 선택, 처음 10개 컬럼으로 중복 행을 제거  
데이터 중복을 정리

In [ ]:
from collections import Counter
col_count = Counter(full_merged.columns)
if any(count > 1 for count in col_count.values()):
    new_cols = []
    seen = {}
    for col in full_merged.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    full_merged.columns = new_cols

hashable_cols = [col for col in full_merged.columns 
                 if full_merged.dtypes[col] in ['int64', 'float64', 'object', 'datetime64[ns]'] 
                 and not full_merged[col].apply(lambda x: isinstance(x, (dict, list))).any()]

full_merged = full_merged.drop_duplicates(subset=hashable_cols[:10])

데이터 클리닝  
'time' 컬럼을 한국 시간으로 변환  
NaN 값을 채우며(지리 정보는 'Unknown', 응답 시간은 중위수, 입력 길이는 0)  
불필요한 컬럼을 삭제  
익명 사용자 플래그('is_anon') 추가, 'count' 컬럼을 정수형으로 변환  
이상치 클리핑

In [ ]:
if 'time' in full_merged.columns:
    full_merged['time_dt'] = pd.to_datetime(full_merged['time'], unit='s', utc=True).dt.tz_convert('Asia/Seoul')

if 'recent_execution_date' in full_merged.columns:
    full_merged['recent_execution_date'] = full_merged['recent_execution_date'].apply(lambda x: x.get('$date') if isinstance(x, dict) else None)
    full_merged['recent_execution_date'] = pd.to_datetime(full_merged['recent_execution_date'])

if 'mp_api_timestamp_ms' in full_merged.columns:
    full_merged.drop('mp_api_timestamp_ms', axis=1, inplace=True)

geo_cols = ['city', 'region', 'country_code', 'timezone']
for col in geo_cols:
    if col in full_merged.columns:
        full_merged[col] = full_merged[col].fillna('Unknown')

if 'response_time_ms' in full_merged.columns:
    median_response = full_merged['response_time_ms'].median()
    full_merged['response_time_ms'] = full_merged['response_time_ms'].fillna(median_response)

if 'input_sentence_length' in full_merged.columns:
    full_merged['input_sentence_length'] = full_merged['input_sentence_length'].fillna(0)

if 'browser' in full_merged.columns:
    full_merged['browser'] = full_merged['browser'].fillna('Unknown')

drop_cols = ['mp_lib', 'mp_api_endpoint', 'mp_loader', 'lib_version', 'mp_processing_time_ms', 'insert_id', 'device_id_event', '_id', '_id_str']
full_merged = full_merged.drop(columns=[col for col in drop_cols if col in full_merged.columns], errors='ignore')

if 'distinct_id' in full_merged.columns:
    full_merged['is_anon'] = full_merged['distinct_id'].str.startswith('$device:')

if 'count' in full_merged.columns:
    full_merged['count'] = full_merged['count'].astype('Int64')

if 'response_time_seconds' in full_merged.columns:
    full_merged['response_time_seconds'] = pd.to_numeric(full_merged['response_time_seconds'], errors='coerce')
    
if 'count' in full_merged.columns:
    percentile_99 = full_merged['count'].quantile(0.99)
    full_merged['count'] = full_merged['count'].clip(upper=percentile_99)

print("Shape after cleaning:", full_merged.shape)
print(full_merged.info())
full_merged.head()

다 같이 필요 없다고 정의한 drop_cols_df2와 drop_cols_df3를 합쳐 중복 제거한 리스트를 만들고, full_merged에서 해당 컬럼들을 삭제

In [ ]:
drop_cols = list(set(drop_cols_df2 + drop_cols_df3))

full_merged = full_merged.drop(columns=[col for col in drop_cols if col in full_merged.columns])

In [ ]:
full_merged.info()

병합된 데이터프레임을 'full_merged.csv' 파일로 저장

In [ ]:
full_merged.to_csv('full_merged.csv', index=False, encoding='utf-8-sig')

In [ ]:
print("Usage Statistics:")
print(user_merged['count'].describe())

In [ ]:
low_usage_pct = (user_merged['count'] <= 1).mean() * 100
print(f"Percentage of users with <=1 usage: {low_usage_pct:.2f}%")

In [ ]:
print("\nTop Browsers:")
print(user_merged['browser'].value_counts(normalize=True).head(10) * 100)

In [ ]:
print("\nTop Operating Systems:")
print(user_merged['os'].value_counts(normalize=True).head(5) * 100)

In [ ]:
print("\nCountry Distribution:")
print(user_merged['country_code'].value_counts(normalize=True).head(5) * 100)

In [ ]:
print("\nEvent Type Distribution:")
event_dist = df3['event_name'].value_counts(normalize=True) * 100
print(event_dist)
    
plt.figure(figsize=(12, 6))
sns.barplot(x=event_dist.index[:10], y=event_dist.values[:10])
plt.title('Top 10 Event Types')
plt.xlabel('Event Name')
plt.ylabel('Percentage')
plt.xticks(rotation=45)
plt.show()

run_count = df3[df3['event_name'] == 'run_paraphrasing'].shape[0]
select_count = df3[df3['event_name'] == 'selected_paraphrasing'].shape[0]
if run_count > 0:
    adoption_rate = (select_count / run_count) * 100
    print(f"Adoption Rate (Selections / Runs): {adoption_rate:.2f}%")

In [ ]:
multi_use_pct = (user_merged['count'] > 1).mean() * 100
print(f"Percentage of users with multiple usages : {multi_use_pct:.2f}%")

In [ ]:
daily_active = full_merged.groupby(full_merged['time_dt'].dt.date)['distinct_id_event'].nunique()
daily_active = daily_active.sort_index()
    
plt.figure(figsize=(12, 6))
daily_active.plot(kind='line', marker='o')
plt.title('Daily Active Users')
plt.xlabel('Date')
plt.ylabel('Unique Users')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()
    
print("Daily Active Users Summary:")
print(daily_active.describe())


In [ ]:
daily_activity = full_merged.groupby(full_merged['time_dt'].dt.date)['distinct_id_event'].nunique().reset_index()
daily_activity['weekday'] = pd.to_datetime(daily_activity['time_dt']).dt.weekday

corr = daily_activity[['weekday', 'distinct_id_event']].corr().iloc[0, 1]
print(f"Correlation between Weekday and Daily Activity: {corr:.2f}")
avg_by_day = daily_activity.groupby('weekday')['distinct_id_event'].mean()
plt.figure(figsize=(8, 5))
sns.barplot(x=avg_by_day.index, y=avg_by_day.values)
plt.title('Average Unique Users by Day of Week')
plt.xlabel('Day of Week (0=Mon, 6=Sun)')
plt.ylabel('Average Unique Users')
plt.show()

In [ ]:
full_merged['hour'] = full_merged['time_dt'].dt.hour
full_merged['weekday'] = full_merged['time_dt'].dt.weekday  
activity_heatmap = full_merged.groupby(['weekday', 'hour'])['distinct_id_event'].nunique().unstack()

plt.figure(figsize=(12, 6))
sns.heatmap(activity_heatmap, cmap='YlGnBu', annot=True, fmt='d')
plt.title('User Activity by Day of Week and Hour')
plt.xlabel('Hour (0-23, Asia/Seoul Time)')
plt.ylabel('Day of Week (0=Mon, 6=Sun)')
plt.show()



In [ ]:
retention_by_os = user_merged.groupby('os')['count'].apply(lambda x: (x > 1).mean() * 100).sort_values(ascending=False)
print("\nRetention Proxy by OS (% multi-use):")
print(retention_by_os)


In [ ]:
retention_by_country = user_merged.groupby('country_code')['count'].apply(lambda x: (x > 1).mean() * 100).sort_values(ascending=False).head(20)
print("\nRetention Proxy by Country (% multi-use):")
print(retention_by_country)

Sentencify.ai 서비스의 사용자 행동 흐름을 분석하여, 사용자가 서비스를 어떻게 시작하고 이탈하는지 파악하기 위해 퍼널 분석(funnel analysis)을 수행  

정의된 이벤트 단계(funnel_stages)별 고유 사용자 수를 계산하고, 데이터프레임으로 만들어 전환율을 계산하고 시각화

pct_change()로 이전 단계 대비 변화율을 구하고, 출력  

사용자 이탈 지점을 파악

In [ ]:
#단계별 사용자 수/전환율 계산

funnel_stages = ['pageview_ad inflow', 'tutorial_start', 'tutorial_complete', 'run_paraphrasing', 'selected_paraphrasing']

funnel_counts = {}
for stage in funnel_stages:
    funnel_counts[stage] = full_merged[full_merged['event_name'] == stage]['distinct_id_event'].nunique()

funnel_df = pd.DataFrame.from_dict(funnel_counts, orient='index', columns=['users']).sort_index(key=lambda x: [funnel_stages.index(s) for s in x])

funnel_df['conversion_rate'] = funnel_df['users'].pct_change().fillna(1) * 100

print(funnel_df)

Funnel Stage(퍼널 스테이지)란 무엇인가?  
"funnel stage"는 마케팅, 제품 분석, UX/UI 분야에서 사용되는 개념으로, 사용자 행동의 순차적 단계(단계별 깔때기)를 의미합니다. 영어로 "funnel"은 "깔때기"를 뜻하며, 사용자가 서비스나 제품을 이용하는 과정에서 단계가 진행될수록 참여자 수가 점점 줄어드는 모양을 비유합니다. (위쪽 넓고 아래쪽 좁은 깔때기처럼)

주요 설명:

개념: 퍼널(funnel)은 사용자가 목표(예: 구매, 가입, 기능 사용)까지 도달하는 여정을 여러 단계로 나눈 모델입니다. 각 "stage(단계)"는 특정 행동이나 이벤트를 나타냅니다.
예: 전자상거래 사이트의 퍼널 - 방문(visit) → 상품 보기(product view) → 장바구니 추가(add to cart) → 결제(purchase).  

이 Notebook에서: Sentencify.ai 서비스의 퍼널 - 광고 유입 페이지뷰('pageview_ad inflow') → 튜토리얼 시작('tutorial_start') → 튜토리얼 완료('tutorial_complete') → paraphrase 실행('run_paraphrasing') → paraphrase 선택('selected_paraphrasing').


왜 중요한가?  
이탈 분석(drop-off): 각 단계에서 사용자 수가 얼마나 줄어드는지(전환율 계산) 봅니다. 이탈이 많은 단계(예: 튜토리얼 시작 후 완료까지 67% 감소)는 문제 지점으로, UX 개선(예: 튜토리얼 간소화) 대상이 됩니다.  

재방문율(retention) 진단: 재방문율이 낮다면 퍼널 초반(첫 방문 후 이탈)에서 원인이 있을 수 있습니다. 퍼널 분석으로 "왜 사용자가 돌아오지 않는가?"를 데이터로 파악.  
비즈니스 가치: 전환율(conversion rate)을 높이면 매출/사용자 유지 증가. 예: 1단계 1000명 → 2단계 500명 (50% 전환) → ... 최종 100명 (전체 10% 전환).  